In [1]:
import pandas as pd
import torch as t
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [3]:
data=pd.read_table("../alloy_data.txt")

In [8]:
print(data.columns.to_list())

['KS1295[%]', '6082[%]', '2024[%]', 'bat-box[%]', '3003[%]', '4032[%]', 'Al', 'Si', 'Cu', 'Ni', 'Mg', 'Mn', 'Fe', 'Cr', 'Ti', 'Zr', 'V', 'Zn', 'Vf_FCC_A1', 'Vf_DIAMOND_A4', 'Vf_AL15SI2M4', 'Vf_AL3X', 'Vf_AL6MN', 'Vf_MG2ZN3', 'Vf_AL3NI2', 'Vf_AL3NI_D011', 'Vf_AL7CU4NI', 'Vf_AL2CU_C16', 'Vf_Q_ALCUMGSI', 'Vf_AL7CU2FE', 'Vf_MG2SI_C1', 'Vf_AL9FE2SI2', 'Vf_AL18FE2MG7SI10', 'eut. frac.[%]', 'eut. T (�C)', 'T_FCC_A1', 'T_DIAMOND_A4', 'T_AL15SI2M4', 'T_AL3X', 'T_AL6MN', 'T_MG2ZN3', 'T_AL3NI2', 'T_AL3NI_D011', 'T_AL7CU4NI', 'T_AL2CU_C16', 'T_Q_ALCUMGSI', 'T_AL7CU2FE', 'T_MG2SI_C1', 'T_AL9FE2SI2', 'T_AL18FE2MG7SI10', 'T(liqu)', 'T(sol)', 'delta_T', 'delta_T_FCC', 'delta_T_Al15Si2M4', 'delta_T_Si', 'CSC', 'YS(MPa)', 'hardness(Vickers)', 'CTEvol(1/K)(20.0-300.0�C)', 'Density(g/cm3)', 'Volume(m3/mol)', 'El.conductivity(S/m)', 'El. resistivity(ohm m)', 'heat capacity(J/(mol K))', 'Therm.conductivity(W/(mK))', 'Therm. diffusivity(m2/s)', 'Therm.resistivity(mK/W)', 'Linear thermal expansion (1/K)(20.0-

In [5]:
data.describe()

,KS1295[%],6082[%],2024[%],bat-box[%],3003[%],4032[%],Al,Si,Cu,Ni,...,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136
count,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,324632.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,16.500000,16.500000,16.500000,16.500000,16.500000,17.500000,91.024760,4.710985,1.633425,0.569050,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,15.276055,15.276055,15.276055,15.276055,15.276055,15.276055,2.835679,2.273711,0.730324,0.329574,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,79.966460,0.716500,0.058500,0.013000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,3.300000,3.300000,3.300000,3.300000,3.300000,4.300000,89.113631,2.873941,1.072590,0.311980,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,13.200000,13.200000,13.200000,13.200000,13.200000,14.200000,91.267046,4.399894,1.554225,0.527800,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,23.100000,23.100000,23.100000,23.100000,23.100000,24.100000,93.164818,6.259213,2.118360,0.782725,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,99.000000,99.000000,99.000000,99.000000,99.000000,100.000000,97.168700,12.695500,4.612500,2.012800,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
input_cols= data.columns.to_list()[:6]
output_cols= data.columns.to_list()[6:70]
input_cols, output_cols

(['KS1295[%]', '6082[%]', '2024[%]', 'bat-box[%]', '3003[%]', '4032[%]'],
 ['Al',
  'Si',
  'Cu',
  'Ni',
  'Mg',
  'Mn',
  'Fe',
  'Cr',
  'Ti',
  'Zr',
  'V',
  'Zn',
  'Vf_FCC_A1',
  'Vf_DIAMOND_A4',
  'Vf_AL15SI2M4',
  'Vf_AL3X',
  'Vf_AL6MN',
  'Vf_MG2ZN3',
  'Vf_AL3NI2',
  'Vf_AL3NI_D011',
  'Vf_AL7CU4NI',
  'Vf_AL2CU_C16',
  'Vf_Q_ALCUMGSI',
  'Vf_AL7CU2FE',
  'Vf_MG2SI_C1',
  'Vf_AL9FE2SI2',
  'Vf_AL18FE2MG7SI10',
  'eut. frac.[%]',
  'eut. T (�C)',
  'T_FCC_A1',
  'T_DIAMOND_A4',
  'T_AL15SI2M4',
  'T_AL3X',
  'T_AL6MN',
  'T_MG2ZN3',
  'T_AL3NI2',
  'T_AL3NI_D011',
  'T_AL7CU4NI',
  'T_AL2CU_C16',
  'T_Q_ALCUMGSI',
  'T_AL7CU2FE',
  'T_MG2SI_C1',
  'T_AL9FE2SI2',
  'T_AL18FE2MG7SI10',
  'T(liqu)',
  'T(sol)',
  'delta_T',
  'delta_T_FCC',
  'delta_T_Al15Si2M4',
  'delta_T_Si',
  'CSC',
  'YS(MPa)',
  'hardness(Vickers)',
  'CTEvol(1/K)(20.0-300.0�C)',
  'Density(g/cm3)',
  'Volume(m3/mol)',
  'El.conductivity(S/m)',
  'El. resistivity(ohm m)',
  'heat capacity(J/(mol K))',
  

In [16]:
cleaned = data[input_cols + output_cols].fillna(0)
cleaned

,KS1295[%],6082[%],2024[%],bat-box[%],3003[%],4032[%],Al,Si,Cu,Ni,...,Density(g/cm3),Volume(m3/mol),El.conductivity(S/m),El. resistivity(ohm m),heat capacity(J/(mol K)),Therm.conductivity(W/(mK)),Therm. diffusivity(m2/s),Therm.resistivity(mK/W),Linear thermal expansion (1/K)(20.0-300.0�C),Technical thermal expansion (1/K)(20.0-300.0�C)
0,0.0,0.0,0.0,0.0,0.0,100.0,83.675000,12.250000,0.900000,1.300000,...,2.65803,0.00001,11302200,8.851450e-08,27.3373,159.046,0.000060,0.006288,0.000024,0.000022
1,0.0,0.0,0.0,0.0,3.3,96.7,84.118850,11.865550,0.874425,1.257100,...,2.65879,0.00001,11414800,8.767350e-08,27.3430,160.429,0.000061,0.006232,0.000024,0.000022
2,0.0,0.0,0.0,0.0,6.6,93.4,84.562700,11.481100,0.848850,1.214200,...,2.65935,0.00001,11489900,8.708070e-08,27.3633,161.346,0.000061,0.006203,0.000024,0.000022
3,0.0,0.0,0.0,0.0,9.9,90.1,85.006550,11.096650,0.823275,1.171300,...,2.66111,0.00001,11566400,8.646380e-08,27.3806,162.105,0.000061,0.006168,0.000024,0.000022
4,0.0,0.0,0.0,0.0,13.2,86.8,85.450400,10.712200,0.797700,1.128400,...,2.66318,0.00001,11650800,8.581640e-08,27.3873,163.127,0.000061,0.006127,0.000024,0.000022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324627,95.7,0.0,0.0,0.0,3.3,1.0,80.533928,12.296200,3.381765,1.946140,...,2.72509,0.00001,10539500,9.491770e-08,27.3289,149.812,0.000056,0.006691,0.000023,0.000021
324628,95.7,0.0,0.0,3.3,0.0,1.0,80.539868,12.297322,3.397440,1.946140,...,2.72496,0.00001,10545800,9.494870e-08,27.3291,149.752,0.000056,0.006688,0.000023,0.000021
324629,95.7,0.0,3.3,0.0,0.0,1.0,80.521553,12.309400,3.379290,1.946965,...,2.72497,0.00001,10558200,9.485710e-08,27.3189,149.799,0.000056,0.006686,0.000023,0.000021
324630,95.7,3.3,0.0,0.0,0.0,1.0,80.358533,12.297025,3.531090,1.946965,...,2.72756,0.00001,10552100,9.486620e-08,27.3293,149.771,0.000056,0.006683,0.000023,0.000021


In [23]:
class MLPNetwork(nn.Module):
    def __init__(self, input_size=len(input_cols), output_size=len(output_cols), hidden_size=64, hidden_layers=2, activation=nn.ReLU):
        super(MLPNetwork, self).__init__()
        self.in_layer = nn.Linear(input_size, hidden_size)
        self.hidden_layers = nn.ModuleList()
        for _ in range(hidden_layers - 1):
            self.hidden_layers.append(nn.Linear(hidden_size, hidden_size))
        self.out_layer = nn.Linear(hidden_size, output_size)
        self.activation = activation()
        self.in_norm = nn.BatchNorm1d(input_size)
        self.out_norm= nn.BatchNorm1d(output_size)
        
    def forward(self, x):
        x = self.in_norm(x)
        x = self.activation(self.in_layer(x))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.out_norm(self.out_layer(x))
        return x
    
import torch.utils.data as data_utils
from tqdm import tqdm

In [27]:
device = "cuda" if t.cuda.is_available() else "mps" if t.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
network = MLPNetwork(
    input_size=len(input_cols),
    output_size=len(output_cols),
    hidden_size=128,
    hidden_layers=4,
    activation=nn.ReLU,
).to(device)
optimizer = optim.Adam(network.parameters(), lr=0.01)
criterion = nn.MSELoss()
network.train()
inputs = t.tensor(cleaned[input_cols].values, dtype=t.float32)
targets = t.tensor(cleaned[output_cols].values, dtype=t.float32)
dataset = data_utils.TensorDataset(inputs, targets)
dataloader = data_utils.DataLoader(dataset, batch_size=128, shuffle=True)
epochs = 10
for epoch in range(epochs):
    gen=tqdm(dataloader)
    for batch_inputs, batch_targets in gen:
        optimizer.zero_grad()
        outputs = network(batch_inputs.to(device))
        loss = criterion(outputs, batch_targets.to(device))
        loss.backward()
        optimizer.step()
        gen.set_description(f"Epoch {epoch+1}/{epochs}")
        gen.set_postfix(loss=loss.item())
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')

Using device: mps


Epoch 1/10: 100%|██████████| 2537/2537 [00:17<00:00, 146.00it/s, loss=2.58e+12]


Epoch 1/10, Loss: 2584731451392.0


Epoch 2/10:  12%|█▏        | 295/2537 [00:02<00:15, 147.03it/s, loss=2.61e+12]


KeyboardInterrupt: 

In [28]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
X = cleaned[input_cols].values
y = cleaned[output_cols].values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)



In [ ]:
model_ensemble: dict[str, lgb.LGBMRegressor] = {}

for i, output_col in enumerate(output_cols):
    model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.01,
        num_leaves=31,
        max_depth=-1,
        random_state=42,
    )
    model.fit(
        X_train,
        y_train[:, i],
        eval_set=[(X_val, y_val[:, i])],
        # early_stopping_rounds=50,
        # verbose=False,
    )
    model_ensemble[output_col] = model
    print(f"Trained model for {output_col}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000611 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 194
[LightGBM] [Info] Number of data points in the train set: 259705, number of used features: 6
[LightGBM] [Info] Start training from score 91.024725
Trained model for Al
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001688 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 194
[LightGBM] [Info] Number of data points in the train set: 259705, number of used features: 6
[LightGBM] [Info] Start training from score 4.711207
Trained model for Si
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000499 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_c